# A.R.I.A — Phase 1: Colab GPU server
Free testing setup for **Artificial Restoration of Individuals' Audio**.
Select **Runtime → Change runtime type → T4 GPU**, then run the numbered cells in order.
Use your matching fine-tuned checkpoint, configuration, and extended tokenizer vocabulary.
The Urdu fine-tune uses **`LANG_TAG=ar`**. This notebook performs no transliteration.

Only the model lives across requests. Reference WAV uploads, latents, and synthesized audio
are processed in RAM and are not written to Drive or the Colab filesystem by this server.
Reference audio necessarily travels to this server for `/latents`; synthesis carries latents
and text per request. No voice profile database or model voice cache is used.

**Secrets:** never paste an API key into a cell. Cell 3 reads `API_KEY` from the environment
or generates a new key and displays it once. Copy that value to the future local backend's
`COLAB_API_KEY`. This one-time display is the requested exception to secret-free output;
clear cell outputs before saving/sharing. Colab's **Edit → Notebook settings → Omit code cell
output when saving this notebook** is enabled in notebook metadata as well.
The submitted notebook contains no outputs or credentials.


In [ ]:
# 1. Install once in a fresh Colab GPU runtime. Keep Colab's matching CUDA torch pair.
import importlib.metadata
import os
import platform
from pathlib import Path
import subprocess
import sys
import urllib.request

if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("Run this notebook in a Linux x86_64 Colab GPU runtime.")
if not ((3, 10) <= sys.version_info[:2] < (3, 15)):
    raise RuntimeError("coqui-tts 0.27.5 requires Python 3.10–3.14.")
try:
    importlib.metadata.version("TTS")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("Legacy TTS conflicts with coqui-tts. Use a fresh runtime.")

os.environ.setdefault("COQUI_TTS_SPEC", "coqui-tts==0.27.5")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    os.environ["COQUI_TTS_SPEC"], "transformers>=4.57,<5",
    "fastapi>=0.115,<1", "uvicorn>=0.30,<1", "python-multipart>=0.0.20,<1",
    "nest_asyncio>=1.6,<2", "soundfile>=0.12,<1", "httpx>=0.27,<1",
])
os.environ.setdefault("CLOUDFLARED_PATH", "/content/cloudflared")
os.environ.setdefault("CLOUDFLARED_DOWNLOAD_URL",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
binary = Path(os.environ["CLOUDFLARED_PATH"])
binary.parent.mkdir(parents=True, exist_ok=True)
urllib.request.urlretrieve(os.environ["CLOUDFLARED_DOWNLOAD_URL"], binary)
binary.chmod(0o700)
subprocess.run([str(binary), "--version"], check=True)
print("Packages installed. If Colab requests a runtime restart, restart and continue at cell 2.")


In [ ]:
# 2. Mount your own Google Drive. Grant access in Colab's standard authorization UI.
import os
from google.colab import drive

os.environ.setdefault("DRIVE_MOUNT", "/content/drive")
drive.mount(os.environ["DRIVE_MOUNT"])


In [ ]:
# 3. Edit non-secret defaults here, or set environment variables before running this cell.
import math
import os
from pathlib import Path
import secrets

if globals().get("server_thread") and server_thread.is_alive():
    raise RuntimeError("Run the shutdown cell before changing settings.")

DEFAULTS = {
    # All three files must be from the same training run, including the 11 added characters.
    "MODEL_PATH": "/content/drive/MyDrive/ARIA/model/model.pth",
    "CONFIG_PATH": "/content/drive/MyDrive/ARIA/model/config.json",
    "VOCAB_PATH": "/content/drive/MyDrive/ARIA/model/vocab.json",
    "LANG_TAG": "ar",  # Urdu fine-tune deliberately uses the Arabic language token.
    "SUPPORTED_LANG_TAGS": "",  # Empty: discover from the matching model configuration.
    "MODEL_VERSION": "urdu-xtts-v1",  # Change on model swaps; used for future enrollment metadata.
    "HOST": "127.0.0.1", "PORT": "8000",
    "TEMPERATURE": "0.65", "SPEED": "1.0",
    "MIN_TEMPERATURE": "0.1", "MAX_TEMPERATURE": "1.5",
    "MIN_SPEED": "0.5", "MAX_SPEED": "2.0",
    "TOP_K": "50", "TOP_P": "0.85", "LENGTH_PENALTY": "1.0",
    "REPETITION_PENALTY": "2.0", "DO_SAMPLE": "true", "NUM_BEAMS": "1",
    "GPT_COND_LEN": "6", "GPT_COND_CHUNK_LEN": "6", "SOUND_NORM_REFS": "false",
    "MIN_REF_SECONDS": "6", "MAX_REF_SECONDS": "30",
    "MIN_REF_SR": "8000", "MAX_REF_SR": "48000", "MAX_REF_CHANNELS": "2",
    "MAX_REQUEST_BYTES": "8388608", "MAX_WAV_BYTES": "6291456",
    "MAX_LATENT_BYTES": "1048576", "MAX_LATENT_TOKENS": "64",
    "MAX_TEXT_CHARS": "180", "MAX_OUTPUT_SECONDS": "60",
    "MAX_INFLIGHT": "3", "BODY_TIMEOUT_SECONDS": "30", "LOCK_TIMEOUT_SECONDS": "90",
    "STARTUP_TIMEOUT_SECONDS": "45", "TUNNEL_TIMEOUT_SECONDS": "60",
    "CLIENT_TIMEOUT_SECONDS": "120",
    "CLOUDFLARED_PATH": "/content/cloudflared", "TUNNEL_PROTOCOL": "http2",
    "SAMPLE_REFERENCE_PATH": "/content/drive/MyDrive/ARIA/sample_reference.wav",
    "SELF_TEST_TEXT": "مجھے پانی چاہیے۔",
}
for name, default in DEFAULTS.items():
    os.environ.setdefault(name, default)

def number(name, cast=float):
    value = cast(os.environ[name])
    if not math.isfinite(value) or value <= 0:
        raise ValueError(f"{name} must be finite and positive")
    return value

def flag(name):
    value = os.environ[name].lower()
    if value not in {"true", "false"}:
        raise ValueError(f"{name} must be true or false")
    return value == "true"

SETTINGS = {name: os.environ[name] for name in DEFAULTS}
for name in (
    "PORT", "TOP_K", "NUM_BEAMS", "GPT_COND_LEN", "GPT_COND_CHUNK_LEN",
    "MIN_REF_SR", "MAX_REF_SR", "MAX_REF_CHANNELS", "MAX_REQUEST_BYTES", "MAX_WAV_BYTES",
    "MAX_LATENT_BYTES", "MAX_LATENT_TOKENS", "MAX_TEXT_CHARS", "MAX_INFLIGHT",
):
    SETTINGS[name] = number(name, int)
for name in (
    "TEMPERATURE", "SPEED", "MIN_TEMPERATURE", "MAX_TEMPERATURE", "MIN_SPEED", "MAX_SPEED",
    "TOP_P", "LENGTH_PENALTY", "REPETITION_PENALTY", "MIN_REF_SECONDS", "MAX_REF_SECONDS",
    "MAX_OUTPUT_SECONDS", "BODY_TIMEOUT_SECONDS", "LOCK_TIMEOUT_SECONDS",
    "STARTUP_TIMEOUT_SECONDS", "TUNNEL_TIMEOUT_SECONDS", "CLIENT_TIMEOUT_SECONDS",
):
    SETTINGS[name] = number(name)
for name in ("DO_SAMPLE", "SOUND_NORM_REFS"):
    SETTINGS[name] = flag(name)
for stem in ("TEMPERATURE", "SPEED"):
    if not SETTINGS[f"MIN_{stem}"] <= SETTINGS[stem] <= SETTINGS[f"MAX_{stem}"]:
        raise ValueError(f"{stem} must be inside its configured bounds")
if not SETTINGS["LANG_TAG"].strip() or SETTINGS["TOP_P"] > 1 or SETTINGS["PORT"] > 65535:
    raise ValueError("Check LANG_TAG, TOP_P, and PORT")
if not (SETTINGS["GPT_COND_CHUNK_LEN"] <= SETTINGS["GPT_COND_LEN"] <= SETTINGS["MAX_REF_SECONDS"]):
    raise ValueError("Require GPT_COND_CHUNK_LEN <= GPT_COND_LEN <= MAX_REF_SECONDS")
if SETTINGS["MIN_REF_SECONDS"] > SETTINGS["MAX_REF_SECONDS"] or SETTINGS["MIN_REF_SR"] > SETTINGS["MAX_REF_SR"]:
    raise ValueError("Reference minimums must not exceed maximums")
if SETTINGS["MAX_WAV_BYTES"] >= SETTINGS["MAX_REQUEST_BYTES"]:
    raise ValueError("MAX_REQUEST_BYTES must leave room for multipart headers")

if not os.environ.get("API_KEY"):
    os.environ["API_KEY"] = secrets.token_urlsafe(32)
    print("Copy to local .env as COLAB_API_KEY (shown once):", os.environ["API_KEY"])
API_KEY = os.environ["API_KEY"]
if len(API_KEY) < 32 or not API_KEY.isascii() or any(c.isspace() for c in API_KEY):
    raise ValueError("API_KEY must contain at least 32 non-whitespace ASCII characters")
# Never print SETTINGS or os.environ: they may contain credentials.
print("Settings ready. Language:", SETTINGS["LANG_TAG"], "Model version:", SETTINGS["MODEL_VERSION"])


In [ ]:
# 4. Load exactly the checkpoint/config/vocabulary specified above, once per runtime.
import gc
import logging
import torch
import torchaudio
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select a Colab T4 GPU runtime.")
for name in ("MODEL_PATH", "CONFIG_PATH", "VOCAB_PATH"):
    if not Path(SETTINGS[name]).is_file():
        raise FileNotFoundError(f"Missing {name}; mount Drive and check the config cell.")
signature = tuple(SETTINGS[k] for k in ("MODEL_PATH", "CONFIG_PATH", "VOCAB_PATH", "LANG_TAG", "MODEL_VERSION"))
if globals().get("loaded_signature") != signature:
    if globals().get("server_thread") and server_thread.is_alive():
        raise RuntimeError("Stop the server before loading another model.")
    if "model" in globals():
        del model
        gc.collect()
        torch.cuda.empty_cache()
    cfg = XttsConfig()
    cfg.load_json(SETTINGS["CONFIG_PATH"])
    cfg.model_args.tokenizer_file = SETTINGS["VOCAB_PATH"]
    model = Xtts.init_from_config(cfg)
    model.load_checkpoint(
        cfg, checkpoint_path=SETTINGS["MODEL_PATH"], vocab_path=SETTINGS["VOCAB_PATH"],
        use_deepspeed=False, strict=True,
    )
    # eval() does not return self in every Coqui release, so do not assign its return value.
    model.cuda().eval()
    loaded_signature = signature

# Derive structural values from the loaded model, never assume base XTTS vocabulary/shapes.
GPT_CHANNELS = int(model.args.gpt_n_model_channels)
SPEAKER_DIM = int(model.args.d_vector_dim)
MAX_MODEL_TEXT_TOKENS = int(model.args.gpt_max_text_tokens)
OUTPUT_SR = int(model.args.output_sample_rate)
logging.getLogger("TTS").setLevel(logging.CRITICAL)
print("Model ready:", torch.cuda.get_device_name(0), "language:", SETTINGS["LANG_TAG"], "output Hz:", OUTPUT_SR)

# Advertise only configured model languages. Optional env restricts this set.
model_languages = set(getattr(cfg, "languages", None) or [SETTINGS["LANG_TAG"]])
requested_languages = {x.strip() for x in SETTINGS["SUPPORTED_LANG_TAGS"].split(",") if x.strip()}
if requested_languages and not requested_languages <= model_languages:
    raise ValueError("SUPPORTED_LANG_TAGS contains tags absent from the model configuration")
SUPPORTED_LANGUAGES = sorted(requested_languages or model_languages)
if SETTINGS["LANG_TAG"] not in SUPPORTED_LANGUAGES:
    raise ValueError("LANG_TAG must be available in the configured language list")
print("Available model language tags:", ", ".join(SUPPORTED_LANGUAGES))

# A concrete identity for checkpoint, config, and vocabulary; never replace these files here.
import hashlib
def _model_file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as model_file:
        for chunk in iter(lambda: model_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()
MODEL_FINGERPRINT = hashlib.sha256("|".join(
    _model_file_hash(SETTINGS[key]) for key in ("MODEL_PATH", "CONFIG_PATH", "VOCAB_PATH")
).encode()).hexdigest()
print("Checkpoint fingerprint:", MODEL_FINGERPRINT)


In [ ]:
# 5. Authenticated, bounded, stateless inference API. No raw request/exception logging.
import asyncio
import base64
import binascii
import gc
import hmac
import io
import json
import logging
import math
import threading
import time
from contextlib import contextmanager

import anyio
import numpy as np
import soundfile as sf
from fastapi import FastAPI, HTTPException, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from pydantic import BaseModel, ConfigDict, Field
from starlette.datastructures import UploadFile
from starlette.formparsers import MultiPartParser

if globals().get("server_thread") and server_thread.is_alive():
    raise RuntimeError("Stop the running server before rebuilding the API.")

logger = logging.getLogger("aria.gpu")
logger.handlers.clear()
logger.addHandler(logging.StreamHandler())
logger.setLevel(logging.INFO)
logger.propagate = False

# Starlette normally spools large uploads to disk. The outer gate caps the entire body,
# and a larger spool threshold keeps every accepted reference upload in RAM.
if hasattr(MultiPartParser, "spool_max_size"):
    MultiPartParser.spool_max_size = SETTINGS["MAX_REQUEST_BYTES"]
else:
    MultiPartParser.max_file_size = SETTINGS["MAX_REQUEST_BYTES"]

api = FastAPI(title="A.R.I.A GPU", docs_url=None, redoc_url=None, openapi_url=None)
gpu_lock = threading.Lock()
SUPPORTED_LANGUAGES = globals().get("SUPPORTED_LANGUAGES", [SETTINGS["LANG_TAG"]])

class RequestGate:
    """Authenticate before parsing; cap chunked bodies and all in-flight HTTP requests."""
    def __init__(self, inner):
        self.inner = inner
        self.active = 0

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            return await self.inner(scope, receive, send)
        headers = {k.lower(): v for k, v in scope["headers"]}
        async def reject(status, detail):
            await JSONResponse({"detail": detail}, status_code=status)(scope, receive, send)
        provided = headers.get(b"x-api-key", b"")
        if not hmac.compare_digest(provided, API_KEY.encode("ascii")):
            return await reject(401, "Missing or invalid x-api-key")
        if self.active >= SETTINGS["MAX_INFLIGHT"]:
            return await reject(429, "GPU server busy; retry shortly")
        try:
            advertised = int(headers.get(b"content-length", b"0"))
        except ValueError:
            return await reject(400, "Invalid Content-Length")
        if advertised < 0:
            return await reject(400, "Invalid Content-Length")
        if advertised > SETTINGS["MAX_REQUEST_BYTES"]:
            return await reject(413, "Request body too large")
        if headers.get(b"content-encoding", b"identity").lower() != b"identity":
            return await reject(415, "Compressed request bodies are unsupported")
        self.active += 1
        body = bytearray()
        try:
            deadline = time.monotonic() + SETTINGS["BODY_TIMEOUT_SECONDS"]
            while True:
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    return await reject(408, "Request body timeout")
                try:
                    message = await asyncio.wait_for(receive(), timeout=remaining)
                except asyncio.TimeoutError:
                    return await reject(408, "Request body timeout")
                if message["type"] == "http.disconnect":
                    return
                chunk = message.get("body", b"")
                if len(body) + len(chunk) > SETTINGS["MAX_REQUEST_BYTES"]:
                    return await reject(413, "Request body too large")
                body.extend(chunk)
                if not message.get("more_body", False):
                    break
            delivered = False
            async def replay():
                nonlocal delivered
                if delivered:
                    return await receive()
                delivered = True
                return {"type": "http.request", "body": bytes(body), "more_body": False}
            await self.inner(scope, replay, send)
        finally:
            body.clear()
            self.active -= 1

@api.exception_handler(RequestValidationError)
async def validation_error(request, exc):
    # FastAPI's default errors can echo the submitted text/latents. Return no input values.
    return JSONResponse({"detail": "Invalid request fields or values"}, status_code=422)

@api.exception_handler(Exception)
async def internal_error(request, exc):
    logger.error(json.dumps({"event": "request_failed", "error_type": type(exc).__name__}))
    return JSONResponse({"detail": "GPU request failed; check runtime/model compatibility"}, status_code=500)

@contextmanager
def gpu_slot():
    if not gpu_lock.acquire(timeout=SETTINGS["LOCK_TIMEOUT_SECONDS"]):
        raise HTTPException(503, "GPU busy; retry shortly")
    try:
        with torch.no_grad():
            yield
    except torch.cuda.OutOfMemoryError:
        raise HTTPException(503, "GPU out of memory; shorten the text or restart the runtime") from None
    finally:
        try:
            gc.collect()
            torch.cuda.empty_cache()
        finally:
            gpu_lock.release()

def encode_latent(tensor):
    array = np.ascontiguousarray(tensor.detach().cpu().numpy(), dtype=np.float32)
    output = io.BytesIO()
    np.save(output, array, allow_pickle=False)
    if output.tell() > SETTINGS["MAX_LATENT_BYTES"]:
        raise HTTPException(500, "Model latent exceeds configured limit")
    return base64.b64encode(output.getvalue()).decode("ascii")

def decode_latent(encoded, kind):
    try:
        raw = base64.b64decode(encoded, validate=True)
        if len(raw) > SETTINGS["MAX_LATENT_BYTES"]:
            raise ValueError("size")
        stream = io.BytesIO(raw)
        version = np.lib.format.read_magic(stream)
        if version == (1, 0):
            shape, fortran, dtype = np.lib.format.read_array_header_1_0(stream)
        elif version == (2, 0):
            shape, fortran, dtype = np.lib.format.read_array_header_2_0(stream)
        else:
            raise ValueError("version")
        # Check the header BEFORE np.load can allocate memory from untrusted dimensions.
        if kind == "gpt":
            valid_shape = (len(shape) == 3 and shape[0] == 1
                and 1 <= shape[1] <= SETTINGS["MAX_LATENT_TOKENS"] and shape[2] == GPT_CHANNELS)
        else:
            valid_shape = shape == (1, SPEAKER_DIM, 1)
        if not valid_shape or dtype.kind != "f" or dtype.itemsize != 4 or fortran:
            raise ValueError("shape or dtype")
        if math.prod(shape) * dtype.itemsize != len(raw) - stream.tell():
            raise ValueError("payload length")
        array = np.load(io.BytesIO(raw), allow_pickle=False)
        if not np.isfinite(array).all():
            raise ValueError("nonfinite")
        return np.ascontiguousarray(array, dtype=np.float32)
    except (ValueError, TypeError, EOFError, OSError, binascii.Error):
        raise HTTPException(422, f"Invalid {kind} latent; re-enroll with this model") from None

MAX_B64 = 4 * ((SETTINGS["MAX_LATENT_BYTES"] + 2) // 3)
MODEL_FINGERPRINT = globals().get("MODEL_FINGERPRINT")

class SynthesisRequest(BaseModel):
    model_fingerprint: str | None = Field(None, pattern=r"^[0-9a-f]{64}$")
    model_config = ConfigDict(extra="forbid", allow_inf_nan=False, strict=True)
    text: str = Field(min_length=1, max_length=SETTINGS["MAX_TEXT_CHARS"])
    language: str = Field(default=SETTINGS["LANG_TAG"], min_length=2, max_length=16)
    gpt_cond_latent: str = Field(min_length=1, max_length=MAX_B64)
    speaker_embedding: str = Field(min_length=1, max_length=MAX_B64)
    temperature: float = Field(default=SETTINGS["TEMPERATURE"],
        ge=SETTINGS["MIN_TEMPERATURE"], le=SETTINGS["MAX_TEMPERATURE"])
    speed: float = Field(default=SETTINGS["SPEED"], ge=SETTINGS["MIN_SPEED"], le=SETTINGS["MAX_SPEED"])

@api.get("/health")
async def health():
    return {"ok": True, "gpu": torch.cuda.get_device_name(0), "lang": SETTINGS["LANG_TAG"], "languages": list(SUPPORTED_LANGUAGES), "model_fingerprint": MODEL_FINGERPRINT}

def reference_latents(raw):
    try:
        with sf.SoundFile(io.BytesIO(raw)) as audio:
            if audio.format not in {"WAV", "WAVEX"}:
                raise ValueError("WAV required")
            sr, channels, frames = audio.samplerate, audio.channels, audio.frames
            if not SETTINGS["MIN_REF_SR"] <= sr <= SETTINGS["MAX_REF_SR"]:
                raise ValueError("sample rate")
            if not 1 <= channels <= SETTINGS["MAX_REF_CHANNELS"]:
                raise ValueError("channels")
            if not SETTINGS["MIN_REF_SECONDS"] <= frames / sr <= SETTINGS["MAX_REF_SECONDS"]:
                raise ValueError("duration")
            samples = audio.read(dtype="float32", always_2d=True)
        mono = samples.mean(axis=1)
        if not np.isfinite(mono).all() or np.max(np.abs(mono)) == 0:
            raise ValueError("nonfinite or silent")
    except (ValueError, RuntimeError, sf.LibsndfileError):
        raise HTTPException(422, "Upload a non-silent WAV within configured duration/rate/channel limits") from None
    # These are XTTS's single-reference conditioning primitives. Avoid torchaudio.load
    # and its filesystem/codec dependencies: soundfile has already decoded the WAV in RAM.
    with gpu_slot():
        reference = gpt = speaker = None
        try:
            reference = torch.from_numpy(np.ascontiguousarray(mono)).unsqueeze(0).to("cuda")
            if SETTINGS["SOUND_NORM_REFS"]:
                reference = reference / reference.abs().max()
            gpt = model.get_gpt_cond_latents(reference, sr,
                length=SETTINGS["GPT_COND_LEN"], chunk_length=SETTINGS["GPT_COND_CHUNK_LEN"])
            speaker = model.get_speaker_embedding(reference, sr)
            return {"gpt_cond_latent": encode_latent(gpt), "speaker_embedding": encode_latent(speaker), "model_fingerprint": MODEL_FINGERPRINT}
        finally:
            del reference, gpt, speaker

@api.post("/latents")
async def latents(request: Request):
    if not request.headers.get("content-type", "").lower().startswith("multipart/form-data"):
        raise HTTPException(415, "Send multipart/form-data with one WAV field named file")
    async with request.form(max_files=1, max_fields=0, max_part_size=SETTINGS["MAX_WAV_BYTES"]) as form:
        upload = form.get("file")
        if len(form) != 1 or not isinstance(upload, UploadFile):
            raise HTTPException(422, "One WAV upload named file is required")
        raw = await upload.read(SETTINGS["MAX_WAV_BYTES"] + 1)
        if len(raw) > SETTINGS["MAX_WAV_BYTES"]:
            raise HTTPException(413, "WAV upload too large")
        # Shielded worker: a client disconnect must not release the GPU lock early.
        return await anyio.to_thread.run_sync(reference_latents, raw, abandon_on_cancel=False)

def synthesize_audio(payload):
    if payload.model_fingerprint is not None and payload.model_fingerprint != MODEL_FINGERPRINT:
        raise HTTPException(409, "Checkpoint mismatch; prepare this voice with the loaded model")
    if payload.language not in SUPPORTED_LANGUAGES:
        raise HTTPException(422, "Language is not supported by this model configuration")
    text = payload.text.strip()
    if not text:
        raise HTTPException(422, "Text must not be blank")
    gpt_array = decode_latent(payload.gpt_cond_latent, "gpt")
    speaker_array = decode_latent(payload.speaker_embedding, "speaker")
    with gpu_slot():
        gpt = speaker = result = None
        try:
            # Same preprocessing and limit used by XTTS's non-streaming inference path.
            if len(model.tokenizer.encode(text.lower(), lang=payload.language.split("-")[0])) >= MAX_MODEL_TEXT_TOKENS:
                raise HTTPException(422, "Too many tokenizer tokens; split text into smaller chunks")
            gpt = torch.from_numpy(gpt_array).to("cuda")
            speaker = torch.from_numpy(speaker_array).to("cuda")
            result = model.inference(
                text=text, language=payload.language, gpt_cond_latent=gpt,
                speaker_embedding=speaker, temperature=payload.temperature, speed=payload.speed,
                top_k=SETTINGS["TOP_K"], top_p=SETTINGS["TOP_P"],
                length_penalty=SETTINGS["LENGTH_PENALTY"], repetition_penalty=SETTINGS["REPETITION_PENALTY"],
                do_sample=SETTINGS["DO_SAMPLE"], num_beams=SETTINGS["NUM_BEAMS"],
                enable_text_splitting=False,  # Local backend owns chunking and ordering.
            )
            wave = np.asarray(result["wav"], dtype=np.float32).reshape(-1)
            if wave.size == 0 or wave.size > int(OUTPUT_SR * SETTINGS["MAX_OUTPUT_SECONDS"]) or not np.isfinite(wave).all():
                raise HTTPException(500, "Model returned invalid or oversized audio")
            output = io.BytesIO()
            sf.write(output, np.clip(wave, -1, 1), OUTPUT_SR, format="WAV", subtype="PCM_16")
            return {"wav_b64": base64.b64encode(output.getvalue()).decode("ascii"), "sr": OUTPUT_SR}
        finally:
            del gpt, speaker, result

@api.post("/synthesize")
def synthesize(payload: SynthesisRequest):
    return synthesize_audio(payload)

app = RequestGate(api)
print("API ready: authenticated GET /health, POST /latents, POST /synthesize")


In [ ]:
# 6. Run Uvicorn in a background thread; discover the free HTTPS quick-tunnel URL.
import queue
import re
import subprocess
import threading
import time
import httpx
import nest_asyncio
import uvicorn

nest_asyncio.apply()

def stop_server():
    process = globals().get("tunnel_process")
    if process is not None:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=5)
        if process.stdout:
            process.stdout.close()
    server = globals().get("uvicorn_server")
    if server is not None:
        server.should_exit = True
    thread = globals().get("server_thread")
    if thread is not None and thread.is_alive():
        thread.join(timeout=15)
        if thread.is_alive():
            raise RuntimeError("GPU work is still finishing; wait before starting another server.")
    os.environ.pop("COLAB_URL", None)

stop_server()
local_url = f'http://{SETTINGS["HOST"]}:{SETTINGS["PORT"]}'
# Suppress library logs that can include raw exception messages, paths, and request data.
for log_name in ("uvicorn", "uvicorn.error", "uvicorn.access", "httpx", "httpcore"):
    logging.getLogger(log_name).disabled = True
uvicorn_server = uvicorn.Server(uvicorn.Config(
    app, host=SETTINGS["HOST"], port=SETTINGS["PORT"], access_log=False,
    log_config=None, proxy_headers=False, server_header=False,
))
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()

try:
    deadline = time.monotonic() + SETTINGS["STARTUP_TIMEOUT_SECONDS"]
    while time.monotonic() < deadline:
        if not server_thread.is_alive():
            raise RuntimeError("Uvicorn stopped. Check for a port conflict or restart the runtime.")
        try:
            check = httpx.get(local_url + "/health", headers={"x-api-key": API_KEY}, timeout=2, trust_env=False)
            if check.status_code == 200:
                break
        except httpx.HTTPError:
            pass
        time.sleep(0.25)
    else:
        raise RuntimeError("Local server did not become ready.")

    tunnel_urls = queue.Queue(maxsize=1)
    tunnel_process = subprocess.Popen([
        SETTINGS["CLOUDFLARED_PATH"], "tunnel", "--url", local_url,
        "--no-autoupdate", "--protocol", SETTINGS["TUNNEL_PROTOCOL"],
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    def read_tunnel_output(process, urls):
        # Drain the pipe for the lifetime of the process; never persist raw tunnel logs.
        try:
            for line in process.stdout:
                match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
                if match:
                    try:
                        urls.put_nowait(match.group(0))
                    except queue.Full:
                        pass
        except (ValueError, OSError):
            pass

    tunnel_reader = threading.Thread(target=read_tunnel_output, args=(tunnel_process, tunnel_urls), daemon=True)
    tunnel_reader.start()
    try:
        public_url = tunnel_urls.get(timeout=SETTINGS["TUNNEL_TIMEOUT_SECONDS"])
    except queue.Empty:
        raise RuntimeError("No tunnel URL received. Rerun cell 6; see troubleshooting.") from None
    # A printed hostname does not guarantee the public route is ready yet.
    deadline = time.monotonic() + SETTINGS["TUNNEL_TIMEOUT_SECONDS"]
    while time.monotonic() < deadline:
        if tunnel_process.poll() is not None:
            raise RuntimeError("cloudflared exited before becoming ready.")
        try:
            check = httpx.get(public_url + "/health", headers={"x-api-key": API_KEY}, timeout=5)
            if check.status_code == 200 and check.json().get("ok") is True:
                break
        except (httpx.HTTPError, ValueError):
            pass
        time.sleep(1)
    else:
        raise RuntimeError("Tunnel route is not ready. Rerun cell 6.")
    os.environ["COLAB_URL"] = public_url
    print("COLAB_URL=" + public_url)
    print("Paste this URL into ARIA Connect; paste the key displayed in cell 3 into the other field.")
except BaseException:
    stop_server()
    raise


In [ ]:
# 7. Exercise the PUBLIC tunnel with a 6–30 second reference WAV you are allowed to use.
# Audio is played from memory. Do not save/share notebook outputs containing voice audio.
from IPython.display import Audio, display

def run_self_test():
    path = Path(SETTINGS["SAMPLE_REFERENCE_PATH"])
    if not path.is_file():
        raise FileNotFoundError("Set SAMPLE_REFERENCE_PATH in cell 3 to your reference WAV.")
    if path.stat().st_size > SETTINGS["MAX_WAV_BYTES"]:
        raise ValueError("Self-test reference exceeds MAX_WAV_BYTES")
    with httpx.Client(base_url=os.environ["COLAB_URL"],
            headers={"x-api-key": API_KEY}, timeout=SETTINGS["CLIENT_TIMEOUT_SECONDS"]) as client:
        def checked(response, expected=200):
            if response.status_code != expected:
                raise RuntimeError(f"Self-test failed with HTTP {response.status_code}; see troubleshooting.")
            return response
        health_data = checked(client.get("/health")).json()
        print("Health:", health_data)
        if not health_data["ok"] or health_data["lang"] != SETTINGS["LANG_TAG"]:
            raise RuntimeError("Health contract or language mismatch")
        checked(client.get("/health", headers={"x-api-key": "invalid"}), expected=401)
        with path.open("rb") as reference:
            latents = checked(client.post("/latents", files={"file": ("reference.wav", reference, "audio/wav")})).json()
        for key in ("gpt_cond_latent", "speaker_embedding"):
            value = np.load(io.BytesIO(base64.b64decode(latents[key], validate=True)), allow_pickle=False)
            assert value.dtype == np.float32 and np.isfinite(value).all()
            print(key + " shape:", value.shape)
        started = time.monotonic()
        result = checked(client.post("/synthesize", json={
            "text": SETTINGS["SELF_TEST_TEXT"], **latents,
            "temperature": SETTINGS["TEMPERATURE"], "speed": SETTINGS["SPEED"],
        })).json()
        wav = base64.b64decode(result["wav_b64"], validate=True)
        samples, sr = sf.read(io.BytesIO(wav))
        assert sr == result["sr"] == OUTPUT_SR and len(samples) > 0
        print(f"Synthesis passed in {time.monotonic() - started:.1f}s; {len(samples) / sr:.1f}s audio at {sr} Hz.")
        display(Audio(data=wav))
    # All request data above is function-local; no voice cache is created.

run_self_test()


## Troubleshooting and operating notes

- **Checkpoint missing/unexpected keys, embedding size mismatch:** confirm all three paths
  refer to the same fine-tune, with the vocabulary containing the 11 Urdu additions.
  Keep strict loading enabled. A trainer-only GPT checkpoint may need export using the
  original training code into a full XTTS inference checkpoint. Do not silently drop keys,
  resize embeddings, use the base vocabulary, or change the tag to `ur`.
- **Checkpoint pickle/safe-globals error:** only load your trusted checkpoint. Use the
  package version used for training via `COQUI_TTS_SPEC`, or export with that environment.
  Do not globally disable PyTorch's safe loading. Back up training artifacts before loading;
  some Coqui versions migrate legacy checkpoint metadata on load.
- **Package conflicts or import errors:** delete the Colab runtime and reconnect, select
  a GPU, then run cell 1 once. Do not install the old `TTS` package alongside `coqui-tts`.
  Restart the runtime if pip replaced a library already imported by Colab. Preserve the
  matched CUDA `torch`/`torchaudio` pair supplied by Colab. `pip check` identifies dependency
  conflicts; changing to another runtime image may be necessary. No actual GPU execution
  is claimed by the local mock tests.
- **Drive authorization / file not found:** run cell 2 interactively, authorize the correct
  Google account, and verify paths under your mounted Drive. A browser blocking the Google
  authorization popup may need popup permission. To remount deliberately, use
  `drive.mount(os.environ['DRIVE_MOUNT'], force_remount=True)`. Never paste OAuth tokens.
- **401:** send `x-api-key` even for `/health`. After a runtime reset, copy the newly generated
  key into the local environment. To rotate deliberately, stop the server, remove `API_KEY`
  from `os.environ`, then rerun cells 3, 5, and 6. Clear the old printed output.
- **413 / 422:** send multipart field `file` with a WAV satisfying configured limits.
  Synthesis needs finite float32 `.npy` arrays (not JSON lists or pickled arrays) from this
  model. Text must fit character AND tokenizer limits; local chunking comes in Phase 2.
- **429 / GPU busy / out of memory:** requests are bounded and both GPU operations share
  one lock. Retry with backoff; shorten text, reduce reference size, or restart the runtime.
  `empty_cache` frees unused allocator blocks; it cannot free live tensors or guarantee
  zeroization of GPU memory. There is no persistent voice store, but RAM is not securely erased.
- **Tunnel URL missing / expired:** rerun cell 6. Quick tunnels need no account and issue a
  different address per session. A preexisting `~/.cloudflared/config.yaml` can prevent quick
  tunnels; use a clean runtime or move your own configuration aside. Do not publish your key.
  Cloudflare is a transport provider for these requests, so this is not a direct private link.
  Free Colab can disconnect and quick tunnels have no uptime guarantee.
- **Changing a setting:** `setdefault` preserves existing environment values. Update with
  `os.environ['NAME'] = 'new non-secret value'`, stop the server, and rerun cells 3–6.
  On a model swap change `MODEL_PATH`, `CONFIG_PATH`, `VOCAB_PATH`, `LANG_TAG`, and
  `MODEL_VERSION`; rerun model loading and re-enroll every voice. Same-shaped old latents
  cannot be identified by the stateless Phase 1 protocol; the local backend will track versions.
- **Shutdown:** run cell 8 when finished. Do not use keepalive tricks to evade free Colab limits.

### Phase 1 acceptance checklist
1. GPU detected; matching checkpoint/config/extended vocab load successfully with `LANG_TAG=ar`.
2. Public `/health` returns `ok`, `gpu`, `lang`; missing/wrong key gives 401.
3. Reference upload yields two finite float32 arrays; self-test speaks the Urdu sentence.
4. Empty/invalid/oversized inputs fail without exposing text, latents, or credentials.
5. Concurrent inference is serialized; generated audio is a playable WAV at the model rate.
6. Stopping/restarting the tunnel works; no user audio/latent files are created by the server.

Sources: [Coqui XTTS implementation](https://github.com/idiap/coqui-ai-TTS/blob/v0.27.5/TTS/tts/models/xtts.py),
[Coqui package](https://pypi.org/project/coqui-tts/0.27.5/),
[Cloudflare quick tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/).


## Long recordings and multiple languages
The local backend accepts long recordings and scans them for a suitable conditioning section (30 seconds by default). Only that mono WAV is uploaded to this thin GPU server; keep its reference duration limit aligned with the backend's MAX_REFERENCE_SECONDS.

The model cell advertises the loaded configuration's language tags through /health. /synthesize accepts an optional language tag and rejects tags outside that list. Urdu continues to use LANG_TAG=ar by default. Set SUPPORTED_LANG_TAGS to a comma-separated subset if you want to restrict the advertised list. Re-run cells 3–6 after changes. The list reflects configuration, not a measured quality guarantee after fine-tuning.


In [ ]:
# 8. OPTIONAL: run only when you want to stop serving (not during initial setup).
stop_server()
print("Tunnel and HTTP server stopped. The loaded model remains ready for cell 6.")
